# 07 · Interactive map of all surf spots

**Purpose.** Show every spot in `data/spots/spots.csv` on an interactive map (zoom, pan, street or satellite view), similar to Google Maps. Click a marker to see the spot profile.

**Inputs:** `data/spots/spots.csv`; optionally `data/resources/cwa_stations_en.csv` (CWA buoys and tide stations).

**Outputs:** the map shown inline below, and `results/figures/surf_spot_map.html`. Open that file in any browser; it also works without VS Code and can be sent to teammates.

**How to read the map**
- **Marker colour = skill level:** green = suits beginners, orange = intermediate and up, red = advanced only.
- **Marker icon = break type:** ☀ beach break, ➤ point break, ◎ reef, 💧 rivermouth.
- **Short blue line** from each marker = the direction the beach faces (the swell must come from roughly that side; offshore wind comes from the opposite side).
- **Search box** (top, next to the zoom buttons): type a spot name from the search results (e.g. *Cijin Beach*), an alternative name (*Sizihwan*) or a spot ID (*NSYSU_BEACH*); pick a suggestion or press Enter, and the map zooms to the spot and opens its profile.
- **Layer menu** (top right): switch between street map (English labels), OpenStreetMap and satellite view, and show/hide break types, CWA buoys and tide stations.

This notebook reads the CSV directly, so it does not need notebooks 00–06.

In [9]:
# ---- Configuration ----
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError("Project root not found. Open the project folder in VS Code (File > Open Folder).")


ROOT = find_project_root()
SPOTS_FILE = ROOT / "data" / "spots" / "spots.csv"
STATIONS_FILE = ROOT / "data" / "resources" / "cwa_stations_en.csv"
OUTPUT_FILE = ROOT / "results" / "figures" / "surf_spot_map.html"

SAMPLE_N = None                 # e.g. 10 = only the first 10 spots
SEED = 42                       # not used; kept for a uniform config across notebooks
MAP_CENTER = [23.7, 120.95]     # roughly the middle of Taiwan
MAP_ZOOM = 8
SHOW_STATIONS = True            # add CWA buoys and tide stations as optional layers
FACING_LINE_KM = 6              # length of the "beach faces" line

## 1. Load the spots
List fields in the CSV are separated by `;`. We split them here only for display.

In [10]:
import json
import math
from html import escape

import folium
import pandas as pd
from folium.plugins import Fullscreen, MiniMap
from IPython.display import display

spots = pd.read_csv(SPOTS_FILE, dtype=str).fillna("")
for col in ["lat", "lon", "orientation_deg"]:
    spots[col] = pd.to_numeric(spots[col], errors="coerce")
spots = spots.dropna(subset=["lat", "lon"])
if SAMPLE_N:
    spots = spots.head(SAMPLE_N)
print(f"{len(spots)} spots loaded")
display(spots[["spot_id", "name", "region", "break_type", "skill_levels"]].head())

35 spots loaded


,spot_id,name,region,break_type,skill_levels
0,WAIAO_BEACH,Wai'ao Beach,northeast,beach,beginner;intermediate;advanced
1,JINZUN_HARBOR,Jinzun Harbor,east,point,intermediate;advanced
2,JIALESHUEI,Jialeshuei,south,beach,beginner;intermediate;advanced
3,CIJIN_BEACH,Cijin Beach,southwest,beach,beginner;intermediate;advanced
4,BAISHAWAN,Baishawan,north,beach,beginner;intermediate;advanced


## 2. Marker style and popup
The popup shows the most useful profile fields and links to the sources, so the map doubles as a quick way to check the corpus.

In [11]:
COMPASS = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
BREAK_ICONS = {"beach": "sun", "point": "location-arrow", "reef": "circle-dot", "rivermouth": "droplet"}   # Font Awesome 6 names
BREAK_LABELS = {"beach": "Beach breaks", "point": "Point breaks", "reef": "Reef breaks", "rivermouth": "Rivermouths"}


def compass(deg):
    return "?" if pd.isna(deg) else COMPASS[int((deg % 360) / 22.5 + 0.5) % 16]


def skill_colour(levels):
    levels = levels.split(";")
    if "beginner" in levels:
        return "green"
    if "intermediate" in levels:
        return "orange"
    return "red"


def popup_html(s):
    sources = "".join(f'<li><a href="{escape(u)}" target="_blank">{escape(u[:60])}</a></li>'
                      for u in s["sources"].split(";") if u.strip())
    rows = [("Region", f"{s['region']} · near {s['nearest_town']}"),
            ("Break / bottom", f"{s['break_type']} · {s['bottom_type']}"),
            ("Faces", f"{compass(s['orientation_deg'])} ({s['orientation_deg']:.0f}°)"),
            ("Swell window", f"{s['swell_dir_min_deg']}° – {s['swell_dir_max_deg']}°"),
            ("Best tide", s["ideal_tide"].replace(";", ", ")),
            ("Level", s["skill_levels"].replace(";", ", ")),
            ("Crowd", s["crowd_level"]),
            ("Hazards", s["hazards"].replace(";", ", ") or "–"),
            ("Facilities", s["facilities"].replace(";", ", ") or "–")]
    table = "".join(f"<tr><td style='color:#57606a;padding-right:8px;vertical-align:top'>{k}</td>"
                    f"<td>{escape(str(v))}</td></tr>" for k, v in rows)
    return (f"<div style='font-family:sans-serif;font-size:12px;width:320px'>"
            f"<h4 style='margin:0 0 4px 0'>{escape(s['name'])}</h4>"
            f"<div style='color:#57606a;margin-bottom:6px'>{escape(s['spot_id'])}</div>"
            f"<table>{table}</table>"
            f"<p style='margin:6px 0'>{escape(s['description'][:400])}{'…' if len(s['description']) > 400 else ''}</p>"
            f"<b>Sources</b><ul style='margin:2px 0 0 16px;padding:0'>{sources}</ul></div>")


def facing_end(lat, lon, deg, km=FACING_LINE_KM):
    dlat = km / 111.0 * math.cos(math.radians(deg))
    dlon = km / (111.0 * math.cos(math.radians(lat))) * math.sin(math.radians(deg))
    return [lat + dlat, lon + dlon]

## 3. Build the map
Three base maps (Esri street map with English labels, OpenStreetMap, Esri satellite imagery), one toggleable layer per break type, a legend, a mini-map and a full-screen button.

In [12]:
m = folium.Map(location=MAP_CENTER, zoom_start=MAP_ZOOM, tiles=None, control_scale=True)
folium.TileLayer(   # English place labels (OpenStreetMap shows local Chinese names in Taiwan)
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Source: Esri, HERE, Garmin, OpenStreetMap contributors", name="Street map (English labels)",
).add_to(m)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", show=False).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Source: Esri, Maxar, Earthstar Geographics", name="Satellite", show=False,
).add_to(m)

layers = {bt: folium.FeatureGroup(name=f"{label} ({(spots['break_type'] == bt).sum()})").add_to(m)
          for bt, label in BREAK_LABELS.items() if (spots["break_type"] == bt).any()}

SEARCH_INDEX = []   # one entry per searchable label (name, alternative names, spot ID)
for _, s in spots.iterrows():
    layer = layers.get(s["break_type"], m)
    marker = folium.Marker(
        [s["lat"], s["lon"]],
        tooltip=f"{s['name']} ({s['break_type']}, {s['skill_levels'].replace(';', '/')})",
        popup=folium.Popup(popup_html(s), max_width=360),
        icon=folium.Icon(color=skill_colour(s["skill_levels"]), icon=BREAK_ICONS.get(s["break_type"], "info"), prefix="fa"),
    ).add_to(layer)
    target = {"marker": marker.get_name(), "layer": layer.get_name(), "lat": s["lat"], "lon": s["lon"], "name": s["name"]}
    SEARCH_INDEX.append({"label": s["name"], **target})
    for alt in [a.strip() for a in s["alt_names"].split(";") if a.strip()] + [s["spot_id"]]:
        if alt.lower() != s["name"].lower():
            SEARCH_INDEX.append({"label": f"{alt} → {s['name']}", **target})
    if pd.notna(s["orientation_deg"]):
        folium.PolyLine([[s["lat"], s["lon"]], facing_end(s["lat"], s["lon"], s["orientation_deg"])],
                        color="#1f6feb", weight=3, opacity=0.8).add_to(layer)

print(f"{len(spots)} markers in {len(layers)} break-type layers")

35 markers in 4 break-type layers


### Optional: CWA buoys and tide stations
These are the stations used to check forecasts. They are hidden by default; switch them on in the layer menu to see which buoy is near which spot.

In [13]:
if SHOW_STATIONS and STATIONS_FILE.exists():
    stations = pd.read_csv(STATIONS_FILE, dtype={"station_id": str}).dropna(subset=["lat", "lon"])
    stations = stations[stations["lat"].between(21, 26.5) & stations["lon"].between(119, 122.5)]   # around Taiwan only
    for stype, label, colour in [("buoy", "CWA buoys", "#8250df"), ("tide", "CWA tide stations", "#6e7781")]:
        group = folium.FeatureGroup(name=f"{label} ({(stations['station_type'] == stype).sum()})", show=False).add_to(m)
        for _, st in stations[stations["station_type"] == stype].iterrows():
            folium.CircleMarker([st["lat"], st["lon"]], radius=5, color=colour, fill=True, fill_opacity=0.9,
                                tooltip=f"{st['station_name_en']} ({st['station_id']})").add_to(group)
    print(f"{len(stations)} CWA stations added (hidden by default)")

74 CWA stations added (hidden by default)


In [14]:
legend = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: white; padding: 10px 12px;
            border-radius: 8px; box-shadow: 0 1px 4px rgba(0,0,0,.3); font: 12px sans-serif; color: #1f2328;">
  <b>Surf spots</b><br>
  <span style="color:#72b026">&#9679;</span> suits beginners<br>
  <span style="color:#f69730">&#9679;</span> intermediate and up<br>
  <span style="color:#d63e2a">&#9679;</span> advanced only<br>
  <span style="color:#1f6feb">&#9473;</span> direction the beach faces<br>
  <span style="color:#57606a">Icons: &#9728; beach &middot; &#10148; point &middot; &#9678; reef &middot; &#128167; rivermouth</span>
</div>"""
m.get_root().html.add_child(folium.Element(legend))
folium.LayerControl(collapsed=False).add_to(m)
MiniMap(toggle_display=True, position="bottomright").add_to(m)
Fullscreen(position="topleft").add_to(m)

### Search box
A text field with suggestions for every spot name, alternative name and spot ID, so a result from the *Surf Spot Search Taiwan* demo can be found on the map directly. The map also opens a spot given in the web address, e.g. `surf_spot_map.html#NSYSU_BEACH`; the demo's *Show on map* links use this. Selecting a suggestion (or pressing Enter) zooms to the spot and opens its profile. If the spot's break-type layer is switched off, it is switched back on.

In [15]:
search_html = """
<div id="spot-search" style="position:absolute; top:10px; left:60px; z-index:1000; background:white; padding:6px 8px;
     border-radius:8px; box-shadow:0 1px 4px rgba(0,0,0,.3); font:13px sans-serif; color:#1f2328;">
  <input id="spot-search-input" list="spot-search-list" placeholder="Search a surf spot, e.g. Cijin Beach"
         style="width:260px; padding:5px 8px; border:1px solid #d0d7de; border-radius:6px; font-size:13px;">
  <button id="spot-search-button" style="padding:5px 10px; border:0; border-radius:6px; background:#1f6feb; color:white; cursor:pointer;">Go</button>
  <datalist id="spot-search-list"></datalist>
  <div id="spot-search-msg" style="color:#a40e26; font-size:12px; margin-top:4px; display:none;"></div>
</div>"""

search_js = """
window.addEventListener("load", function () {   // run after the map has been created
  var MAP = %(map)s;
  var INDEX = %(index)s;
  var input = document.getElementById("spot-search-input");
  var list = document.getElementById("spot-search-list");
  var msg = document.getElementById("spot-search-msg");
  INDEX.forEach(function (e) { var o = document.createElement("option"); o.value = e.label; list.appendChild(o); });
  L.DomEvent.disableClickPropagation(document.getElementById("spot-search"));
  L.DomEvent.disableScrollPropagation(document.getElementById("spot-search"));

  function find(q) {
    q = q.trim().toLowerCase();
    if (!q) return null;
    return INDEX.find(function (e) { return e.label.toLowerCase() === q; })
        || INDEX.find(function (e) { return e.label.toLowerCase().startsWith(q); })
        || INDEX.find(function (e) { return e.label.toLowerCase().indexOf(q) >= 0; }) || null;
  }
  function go() {
    var e = find(input.value);
    if (!e) { msg.textContent = 'No surf spot found for: ' + input.value; msg.style.display = 'block'; return; }
    msg.style.display = "none";
    input.value = e.name;
    var layer = window[e.layer], marker = window[e.marker];
    if (layer && !MAP.hasLayer(layer)) { MAP.addLayer(layer); }
    MAP.once("moveend", function () { marker.openPopup(); });
    MAP.flyTo([e.lat, e.lon], 13, {duration: 1.2});
  }
  input.addEventListener("change", function () { if (find(input.value)) go(); });
  input.addEventListener("keydown", function (ev) { if (ev.key === "Enter") { ev.preventDefault(); go(); } });
  document.getElementById("spot-search-button").addEventListener("click", go);
  if (window.location.hash.length > 1) {   // e.g. surf_spot_map.html#NSYSU_BEACH (used by the demo's "Show on map" links)
    input.value = decodeURIComponent(window.location.hash.slice(1));
    setTimeout(go, 300);
  }
});
""" % {"map": m.get_name(), "index": json.dumps(SEARCH_INDEX, ensure_ascii=False)}

m.get_root().html.add_child(folium.Element(search_html))
m.get_root().script.add_child(folium.Element(search_js))
print(f"Search box added: {len(SEARCH_INDEX)} searchable labels for {len(spots)} spots")

Search box added: 115 searchable labels for 35 spots


## 4. Show and save
The saved HTML file is self-contained apart from the map tiles, which are loaded from the internet when it is opened.

In [16]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
m.save(str(OUTPUT_FILE))
print(f"Saved: {OUTPUT_FILE.relative_to(ROOT)}  (open it in a browser for a full-screen map)")
m

Saved: results\figures\surf_spot_map.html  (open it in a browser for a full-screen map)
